### We need Python 3.10

In [ ]:
!python --version

Python 3.10.12


In [2]:
import optiver2023
env = optiver2023.make_env()
iter_test = env.iter_test()

In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/optiver-trading-at-the-close/public_timeseries_testing_util.py
/kaggle/input/optiver-trading-at-the-close/train.csv
/kaggle/input/optiver-trading-at-the-close/example_test_files/sample_submission.csv
/kaggle/input/optiver-trading-at-the-close/example_test_files/revealed_targets.csv
/kaggle/input/optiver-trading-at-the-close/example_test_files/test.csv
/kaggle/input/optiver-trading-at-the-close/optiver2023/competition.cpython-310-x86_64-linux-gnu.so
/kaggle/input/optiver-trading-at-the-close/optiver2023/__init__.py


In [4]:
INPUT_FILE_PATH="/kaggle/input/optiver-trading-at-the-close/"
OUTPUT_FILE_PATH="/kaggle/working/"
SUBMISSIONS_PATH=OUTPUT_FILE_PATH+'submissions'
if not os.path.exists(SUBMISSIONS_PATH):
   os.makedirs(SUBMISSIONS_PATH)
   print("The new directory is created!")
else:
   print(f"{SUBMISSIONS_PATH} Directory already exists!")

The new directory is created!


In [5]:
data = pd.read_csv(INPUT_FILE_PATH+'train.csv')

validation_set = data[data["date_id"] > 400] 
validation_set.to_csv(OUTPUT_FILE_PATH+'validation_set.csv', index=False)

train_set = data[data["date_id"] <= 400] 
train_set.to_csv(OUTPUT_FILE_PATH+'train_set.csv', index=False)

In [6]:
train = pd.read_csv(OUTPUT_FILE_PATH+'train_set.csv')
validation = pd.read_csv(OUTPUT_FILE_PATH+'validation_set.csv')
test = pd.read_csv(INPUT_FILE_PATH+'example_test_files/test.csv')

In [7]:
DATASETS = [train, validation, test]
MODELS = set() # use MODELS.add() to add items

#### Add normalized columns 

In [8]:
scaler = StandardScaler()
COLUMNS = ['imbalance_size', 'matched_size', 'bid_size', 'ask_size']

def normalize(DATASET, COLUMN):
    DATASET[COLUMN+"_norm"] = scaler.fit_transform(DATASET[[COLUMN]])

for DATASET in DATASETS:
    for COLUMN in COLUMNS:
        normalize(DATASET, COLUMN)

### Linear Regression
We all know that Linear Regression is a terrible fit. <br/>
So let's all just *pretend* it is an interesting model to check out and use the output on the test-set and API. <br/><br/>
This model is for research purposess only.

#### Deal with NaN values
So there are different ways to deal with NaN values, one is to delete them. But I'd rather not, thus:
- since there arent that many NaN values, just replace them with mean of column

In [9]:
imp_x = SimpleImputer(missing_values=np.nan, strategy='mean')
imp_y = SimpleImputer(missing_values=np.nan, strategy='mean')

def configure_dataset(DATASET, BASE, TARGET_COLUMN):
    print("Configuring datasets...")
    print(f"Target column: {TARGET_COLUMN}")
    y_DATASET = np.empty((len(DATASET), 1))
    
    # Extract target variable as numpy array
    if TARGET_COLUMN:
        y_DATASET = np.array(DATASET[TARGET_COLUMN].values)
        
    X_DATASET = np.array(DATASET[BASE])       
    
    
    if len(X_DATASET) == len(train):
        print(f"Dataset size: {len(X_DATASET)} \timpute dataset")
        imp_x.fit(X_DATASET)
        y_DATASET = imp_y.fit_transform(y_DATASET.reshape(-1, 1)).ravel() #only fill target values for train set.
    
    X_DATASET = imp_x.transform(X_DATASET)
    
    if TARGET_COLUMN:
        return X_DATASET, y_DATASET
    else:
        return X_DATASET

In [10]:
def linear_regression_fit(TARGET_COLUMN, BASE, BASE_NAME):    
    MODEL='linear_regression'+'_'+BASE_NAME
    
    print(f"Start fitting model: {MODEL}...\n")     
    
    # Create sets
    X_train, y_train = configure_dataset(train, BASE, TARGET_COLUMN)
    
    reg = LinearRegression().fit(X_train, y_train)    
    
    print(f"End fitting model: {MODEL}\n")
    print(f"score: {reg.score(X_train, y_train)}")
    print(f"coef_: {reg.coef_}")
    print(f"intercept_: {reg.intercept_}\n\n\n")
    
    return reg

In [11]:
def linear_regression_predict(DATASET, BASE, FITTED_MODEL,BASE_NAME):
    # Use the model output to predict validation prediction values        
    
    MODEL='lin_regr'+'_'+BASE_NAME
    print(f"Start predicting model: {MODEL} with: BASE: {BASE_NAME}...\n")
    
    X = configure_dataset(DATASET, BASE, False)
    
    DATASET[MODEL+'_prediction'] = FITTED_MODEL.predict(X)    
    
    MODELS.add(MODEL)
    print(f"End predicting model: {MODEL} with: BASE: {BASE_NAME}\n\n\n")

    return DATASET

In [12]:
base = ['wap','seconds_in_bucket' ,'reference_price', 'imbalance_size', 'imbalance_buy_sell_flag', 'matched_size', 'bid_price'] 
base_norm = ['wap','seconds_in_bucket' ,'reference_price', 'imbalance_size_norm', 'imbalance_buy_sell_flag', 'matched_size_norm', 'bid_price']

model_lr = linear_regression_fit('target', base, "base")
model_lr_norm = linear_regression_fit('target', base_norm, "base_norm")

Start fitting model: linear_regression_base...

Configuring datasets...
Target column: target
Dataset size: 4357980 	impute dataset
End fitting model: linear_regression_base

score: 0.019470059996827715
coef_: [-4.41601001e+03 -6.33054112e-04  2.77208450e+03 -1.69317314e-09
 -1.56923895e-01 -1.36311859e-10  1.52385589e+03]
intercept_: 120.60537003926122



Start fitting model: linear_regression_base_norm...

Configuring datasets...
Target column: target
Dataset size: 4357980 	impute dataset
End fitting model: linear_regression_base_norm

score: 0.019470059998868305
coef_: [-4.41601004e+03 -6.33056819e-04  2.77208449e+03 -3.39600805e-02
 -1.56923882e-01 -1.84900906e-02  1.52385589e+03]
intercept_: 120.58989359409763





#### Predict Linear Regression

In [13]:
# validation set
pred_val_lr = linear_regression_predict(validation, base, model_lr, "base")
pred_val_lr_norm = linear_regression_predict(validation, base_norm, model_lr_norm, "base_norm")

Start predicting model: lin_regr_base with: BASE: base...

Configuring datasets...
Target column: False
End predicting model: lin_regr_base with: BASE: base



Start predicting model: lin_regr_base_norm with: BASE: base_norm...

Configuring datasets...
Target column: False
End predicting model: lin_regr_base_norm with: BASE: base_norm





In [14]:
# test set
pred_test_lr = linear_regression_predict(test, base, model_lr, "base")
pred_test_lr_norm = linear_regression_predict(test, base_norm, model_lr_norm, "base_norm")

Start predicting model: lin_regr_base with: BASE: base...

Configuring datasets...
Target column: False
End predicting model: lin_regr_base with: BASE: base



Start predicting model: lin_regr_base_norm with: BASE: base_norm...

Configuring datasets...
Target column: False
End predicting model: lin_regr_base_norm with: BASE: base_norm





### Evaluation on validation set
Our evaluation metric is Mean Absolute Error (MAE). The formula is given by:

MAE=1nn∑i=1 |yi−xi| 
Where: n is the total number of data points.
yi  is the predicted value for the  ith  data point.
xi  is the observed value for the  ith  data point.

In [15]:
MAE_DICT = {}
def evaluation(MODELS):
    for MODEL in MODELS:
        mae = (validation[MODEL+'_prediction'] - validation['target']).abs().mean()
        MAE_DICT.update({MODEL : mae})

evaluation(MODELS)

SORTED_MAE_DICT=dict(sorted(MAE_DICT.items(), key=lambda item: item[1])) 

for key, value in SORTED_MAE_DICT.items():
    print(f"{key}: {value}")

lin_regr_base: 5.977291819752582
lin_regr_base_norm: 5.977318690571634


In [16]:
fixed_columns = ['row_id','time_id']
def save_predictions_to_file(MODELS):    
    print("Saving models to file...\n")
    for MODEL in MODELS:
        PREDICTION_COLUMN = MODEL+'_prediction'
        columns = [MODEL+'_prediction']
        columns.extend(fixed_columns)
        columns.reverse()
        #print(f"columns: {columns}")
        file_name = OUTPUT_FILE_PATH+'/submissions/'+PREDICTION_COLUMN+'.csv'
        
        #print(test[columns].rename(columns={PREDICTION_COLUMN:'target'}, inplace=True).head())
        test[columns].rename(columns={PREDICTION_COLUMN:"target"}).to_csv(file_name, index=False, header=True)        
        print(f"Model: {MODEL} to file: {file_name}")
    print("\n\nFinished saving models")
        
save_predictions_to_file(MODELS)  

Saving models to file...

Model: lin_regr_base to file: /kaggle/working//submissions/lin_regr_base_prediction.csv
Model: lin_regr_base_norm to file: /kaggle/working//submissions/lin_regr_base_norm_prediction.csv


Finished saving models


# ----------------------predict the models----------------

In [17]:
def submit(MODELS):        
    for MODEL in MODELS:
        if MODEL == 'lin_regr_base_norm':
            print(f"Submitting model {MODEL}...")
            
            PREDICTION_COLUMN = MODEL+'_prediction'
            
            counter = 0
            for (test, revealed_targets, sample_prediction) in iter_test:
                if counter == 0:
                    print(f"Counter: {counter}")
                    print(f"test:\n {test.head(3)}\n\n")
                    print(f"revealed_targets:\n {revealed_targets.head(3)}\n\n")
                    print(f"sample_prediction:\n {sample_prediction.head(3)}\n\n")

                for COLUMN in COLUMNS:
                    normalize(test, COLUMN)
                    
                predictions = linear_regression_predict(test, base_norm, model_lr_norm, "base_norm")                
                sample_prediction['target'] = predictions[PREDICTION_COLUMN]                
                
                env.predict(sample_prediction)
                
                counter += 1
            print(f"Submitted Model: {MODEL}\n\n")
        else:
            print(f"Skip submitting model {MODEL}...\n")
    
    
    print("\n\nFinished submitting models")
        
submit(MODELS) 

Skip submitting model lin_regr_base...

Submitting model lin_regr_base_norm...
This version of the API is not optimized and should not be used to estimate the runtime of your code on the hidden test set.
Counter: 0
test:
    stock_id  date_id  seconds_in_bucket  imbalance_size  \
0         0      478                  0      3753451.43   
1         1      478                  0       985977.11   
2         2      478                  0       599128.74   

   imbalance_buy_sell_flag  reference_price  matched_size  far_price  \
0                       -1         0.999875   11548975.43        NaN   
1                       -1         1.000245    3850033.97        NaN   
2                        1         1.000584    4359198.25        NaN   

   near_price  bid_price  bid_size  ask_price  ask_size  wap   row_id  \
0         NaN   0.999875  22940.00   1.000050   9177.60  1.0  478_0_0   
1         NaN   0.999940   1967.90   1.000601  19692.00  1.0  478_0_1   
2         NaN   0.999918   4488.2

In [18]:
for MODEL in MODELS:
    print(MODEL)

lin_regr_base
lin_regr_base_norm
